# Support Vector Machines (SVM)

SVMs find the optimal hyperplane that maximizes the margin between classes.

1. **Linear SVM** - Maximum margin classifier
2. **Kernel Trick** - RBF, polynomial kernels for nonlinear boundaries
3. **Hyperparameter Tuning** - C (regularization) and gamma (kernel width)

**Dataset**: Digits (multi-class classification)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits, make_moons
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")

## 1. Visualizing the Kernel Trick

Linear SVMs can only separate linearly separable data. The kernel trick projects data into higher dimensions where a linear separator exists.

In [ ]:
# Create nonlinear dataset
X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

kernels = ["linear", "rbf", "poly"]
for ax, kernel in zip(axes, kernels):
    svm = SVC(kernel=kernel, C=1.0, gamma="scale", degree=3)
    svm.fit(X_moons, y_moons)
    
    # Create mesh for decision boundary
    xx, yy = np.meshgrid(
        np.linspace(X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5, 200),
        np.linspace(X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5, 200),
    )
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="RdBu")
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap="RdBu", edgecolors="k", s=20)
    ax.set_title(f"{kernel.upper()} kernel (acc={svm.score(X_moons, y_moons):.2f})")

plt.tight_layout()
plt.show()

## 2. SVM on Digits Dataset

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# SVM requires feature scaling
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=10, gamma="scale", random_state=42)),
])
pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)
print(f"Accuracy: {pipe.score(X_test, y_test):.4f}")
print("\n" + classification_report(y_test, y_pred))

In [ ]:
# Hyperparameter tuning: C and gamma
param_grid = {
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": ["scale", 0.001, 0.01, 0.1],
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)

print(f"Best params: {grid.best_params_}")
print(f"Best CV accuracy: {grid.best_score_:.4f}")
print(f"Test accuracy: {grid.score(X_test, y_test):.4f}")

## Key Takeaways

1. **Always scale features** - SVM is sensitive to feature magnitudes
2. **RBF kernel is the default choice** - works well for most problems
3. **C controls regularization** - high C = less regularization = tighter fit
4. **Gamma controls decision boundary smoothness** - high gamma = complex boundary
5. **SVMs don't scale well** to very large datasets (>100k samples) - use SGD or kernel approximations